# 🧬 Clasificación de Tumores — Gene Expression RNA-Seq
**Dataset:** TCGA PANCAN HiSeq — 801 muestras × 20,531 genes  
**Clases:** BRCA, KIRC, COAD, LUAD, PRAD  
**Pipeline:** Carga → Preprocesamiento → PCA → Clasificación → Evaluación

## 1. Instalación y carga de librerías

In [ ]:
# Instalar el paquete de UCI si no lo tienes
!pip install ucimlrepo -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

# Seed para reproducibilidad
SEED = 42
np.random.seed(SEED)

print('✅ Librerías cargadas correctamente')

## 2. Carga del dataset

In [ ]:
# Descarga automática desde UCI
dataset = fetch_ucirepo(id=401)

X = dataset.data.features   # (801, 20531)
y = dataset.data.targets.squeeze()  # (801,)

print(f'Shape features: {X.shape}')
print(f'Shape target:   {y.shape}')
print(f'\nDistribución de clases:')
print(y.value_counts())

## 3. Análisis exploratorio rápido (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de clases
counts = y.value_counts()
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Distribución de tipos de tumor', fontsize=13)
axes[0].set_ylabel('Número de muestras')
axes[0].set_xlabel('Tipo de tumor')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Distribución de expresión génica (muestra de 500 genes aleatorios)
sample_genes = X.iloc[:, :500].values.flatten()
axes[1].hist(sample_genes, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribución de niveles de expresión\n(primeros 500 genes)', fontsize=13)
axes[1].set_xlabel('Nivel de expresión')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA completado')

## 4. Preprocesamiento

In [ ]:
# Codificar etiquetas
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Clases codificadas:', dict(zip(le.classes_, le.transform(le.classes_))))

# Verificar valores nulos
print(f'\nValores nulos: {X.isnull().sum().sum()}')

# Split estratificado (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc
)

# Escalar (estandarización)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'\nTrain: {X_train_scaled.shape} | Test: {X_test_scaled.shape}')
print('✅ Preprocesamiento completado')

## 5. Reducción de dimensionalidad con PCA

In [ ]:
# PCA para analizar varianza explicada
pca_full = PCA(random_state=SEED)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Varianza acumulada
axes[0].plot(cumvar[:200], color='steelblue', linewidth=2)
axes[0].axhline(0.90, color='orange', linestyle='--', label=f'90% → {n_90} componentes')
axes[0].axhline(0.95, color='red', linestyle='--', label=f'95% → {n_95} componentes')
axes[0].set_xlabel('Número de componentes principales')
axes[0].set_ylabel('Varianza explicada acumulada')
axes[0].set_title('Varianza explicada por PCA', fontsize=13)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Proyección 2D
pca_2d = PCA(n_components=2, random_state=SEED)
X_2d = pca_2d.fit_transform(X_train_scaled)
colors_map = {0:'#4C72B0', 1:'#DD8452', 2:'#55A868', 3:'#C44E52', 4:'#8172B2'}
for cls in np.unique(y_train):
    mask = y_train == cls
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    label=le.classes_[cls], alpha=0.7,
                    color=colors_map[cls], s=30)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Proyección PCA 2D — Train', fontsize=13)
axes[1].legend(title='Tumor')

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n📊 Componentes para 90% varianza: {n_90}')
print(f'📊 Componentes para 95% varianza: {n_95}')
print('✅ Análisis PCA completado')

In [ ]:
# Aplicar PCA con N componentes que explican 95% de varianza
N_COMPONENTS = n_95

pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f'Dimensiones originales: {X_train_scaled.shape[1]}')
print(f'Dimensiones tras PCA:   {X_train_pca.shape[1]}')
print(f'Reducción: {(1 - N_COMPONENTS/X_train_scaled.shape[1])*100:.1f}%')

## 6. Entrenamiento y comparación de clasificadores

In [ ]:
# Definir clasificadores a comparar
classifiers = {
    'SVM (RBF)':         SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED),
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

print('Evaluando clasificadores con validación cruzada (5-fold)...\n')
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_train_pca, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f'{name:<25} Acc: {scores.mean():.4f} ± {scores.std():.4f}')

print('\n✅ Evaluación completada')

In [ ]:
# Visualizar comparación de modelos
fig, ax = plt.subplots(figsize=(9, 5))

names = list(results.keys())
means = [results[n].mean() for n in names]
stds  = [results[n].std()  for n in names]

bars = ax.barh(names, means, xerr=stds, color=['#4C72B0','#55A868','#DD8452'],
               capsize=5, alpha=0.85, edgecolor='white')
ax.set_xlabel('Accuracy (CV 5-fold)')
ax.set_title('Comparación de clasificadores\n(datos reducidos con PCA)', fontsize=13)
ax.set_xlim(0.85, 1.01)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.001, bar.get_y() + bar.get_height()/2,
            f'{mean:.4f}', va='center', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Evaluación del mejor modelo en Test

In [ ]:
# Entrenar el mejor modelo (SVM suele ganar en este dataset)
# Cambia aquí si otro modelo ganó en la comparación
best_clf = SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)
best_clf.fit(X_train_pca, y_train)

y_pred = best_clf.predict(X_test_pca)

print('=== RESULTADOS EN TEST SET ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Matriz de Confusión — SVM (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Importancia de componentes principales

In [ ]:
# Top genes más influyentes en PC1 y PC2
gene_names = X.columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, ax in enumerate(axes):
    loadings = pca.components_[i]
    top_idx = np.argsort(np.abs(loadings))[-20:][::-1]
    top_genes = [gene_names[j] for j in top_idx]
    top_vals  = loadings[top_idx]
    
    colors_bar = ['#4C72B0' if v > 0 else '#C44E52' for v in top_vals]
    ax.barh(top_genes[::-1], top_vals[::-1], color=colors_bar[::-1], alpha=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Top 20 genes — PC{i+1}', fontsize=12)
    ax.set_xlabel('Loading')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('pca_loadings.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Pipeline completo finalizado!')

## 9. Próximos pasos sugeridos

| Mejora | Descripción |
|--------|-------------|
| **UMAP** | Reducción no lineal — suele dar mejor separación visual que PCA |
| **Selección de features** | `SelectKBest` o `VarianceThreshold` para filtrar genes poco informativos |
| **Hyperparameter tuning** | `GridSearchCV` o `RandomizedSearchCV` para optimizar C, gamma en SVM |
| **Ensemble** | Combinar SVM + RF con `VotingClassifier` |
| **XGBoost / LightGBM** | Probar boosting sobre PCA features |